In [ ]:

import csv
import time
import requests
import urllib.robotparser

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE_URL = "https://www.heraldo.es/"
SALIDA = "titulares_heraldo.csv"

# Identificación transparente del scraper
HEADERS = {
    "User-Agent": "HeraldoTitularesBot/1.0 (uso personal)"
}


def comprobar_robots():
    """Comprueba las reglas de rastreo publicadas por el sitio."""
    robots_url = urljoin(BASE_URL, "/robots.txt")
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)

    try:
        rp.read()
        if not rp.can_fetch(HEADERS["User-Agent"], BASE_URL):
            print("El archivo robots.txt no permite rastrear la portada.")
            return False
        return True
    except Exception as e:
        print(f"No se pudo comprobar robots.txt: {e}")
        print("Se detiene el scraper por precaución.")
        return False


def extraer_titulares():
    if not comprobar_robots():
        return []

    try:
        respuesta = requests.get(
            BASE_URL,
            headers=HEADERS,
            timeout=20
        )

        if respuesta.status_code == 403:
            print("Error 403: el servidor ha denegado el acceso.")
            print("No se intentará eludir la restricción.")
            return []

        respuesta.raise_for_status()

    except requests.RequestException as e:
        print(f"Error al descargar la página: {e}")
        return []

    soup = BeautifulSoup(respuesta.text, "html.parser")
    resultados = []
    urls_vistas = set()

    # Busca enlaces que estén dentro de titulares
    for encabezado in soup.find_all(["h1", "h2", "h3", "h4"]):
        enlace = encabezado.find("a", href=True)

        # Algunos titulares tienen el enlace en un elemento superior
        if enlace is None:
            enlace = encabezado.find_parent("a", href=True)

        if enlace is None:
            continue

        titular = " ".join(enlace.get_text(" ", strip=True).split())
        url = urljoin(BASE_URL, enlace["href"])
        parsed = urlparse(url)

        # Evita enlaces externos y titulares vacíos
        if parsed.netloc not in {"heraldo.es", "www.heraldo.es"}:
            continue

        if len(titular) < 15:
            continue

        if url in urls_vistas:
            continue

        urls_vistas.add(url)
        resultados.append({
            "titular": titular,
            "url": url
        })

    return resultados


def guardar_csv(titulares):
    with open(SALIDA, "w", newline="", encoding="utf-8-sig") as archivo:
        campos = ["titular", "url"]
        escritor = csv.DictWriter(archivo, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(titulares)

    print(f"\nGuardados {len(titulares)} titulares en {SALIDA}")


def main():
    titulares = extraer_titulares()

    if not titulares:
        print("No se han encontrado titulares.")
        return

    for i, noticia in enumerate(titulares, start=1):
        print(f"{i}. {noticia['titular']}")
        print(f"   {noticia['url']}")

    guardar_csv(titulares)


main()


1. Aragón convoca 900 plazas en las oposiciones de 2027 para profesores de secundaria y FP
   https://www.heraldo.es/noticias/aragon/2026/10/20/aragon-convoca-900-plazas-las-oposiciones-profesores-secundaria-fp-2027-estas-son-por-especialidades-2061106.html
2. La persecución y muerte de un jabalí en Zaragoza obliga a tomar medidas: "No puedes enfrentarte al animal"
   https://www.heraldo.es/noticias/aragon/zaragoza/2026/10/19/persecucion-muerte-jabali-pleno-centro-zaragoza-obliga-tomar-medidas-no-puedes-enfrentarte-animal-2060748.html
3. La indignación de una madre tras cancelarse la operación de su hijo de 4 años: "Nos han avisado ahora y era mañana"
   https://www.heraldo.es/noticias/aragon/2026/10/20/indignacion-una-madre-zaragoza-tras-cancelarse-operacion-su-hijo-4-anos-nos-acaban-avisar-operaban-manana-2061151.html
4. Revolut se cae e informa sobre el estado del dinero y los datos
   https://www.heraldo.es/noticias/economia/2026/10/20/comunicado-urgente-revolut-cae-informa-dinero-